## LANL Comprehensive, Multi-Source Cyber-Security Events — Silver ETL

**Reads from:** `workspace.bronze_lanl.*`  
**Writes to:** `workspace.silver_lanl.*`

### Transforms applied
| Silver Table | Source | Key Transforms |
|---|---|---|
| `auth` | bronze `auth` | Parse user@domain → user + domain, `?` → NULL, add `is_failed` flag, time buckets |
| `dns` | bronze `dns` | `?` → NULL, time buckets |
| `flows` | bronze `flows` | `?` → NULL, classify well-known ports → `service_type`, time buckets |
| `proc` | bronze `proc` | Parse user@domain → user + domain, `?` → NULL, `start_end` → `is_start` boolean, time buckets |
| `redteam` | bronze `redteam` | Parse user@domain → user + domain |
| `auth_labeled` | silver `auth` + `redteam` | Left-join auth with redteam → `is_compromised` flag for ML |

---

### Table of Contents

1. [Setup & Helpers](#1-setup--helpers) — imports, shared UDFs
2. [Silver Table Transforms](#2-silver-table-transforms) — clean, enrich, standardize
3. [Labeled Dataset Construction](#3-labeled-dataset-construction) — join auth with red-team ground truth
4. [Validation](#4-validation) — row-count sanity checks

---

## 1. Setup & Helpers

Shared helper functions used across all silver transforms:
* **`nullify_question_marks`** — replaces LANL's `?` sentinel with proper SQL `NULL`
* **`parse_user_domain`** — splits `USER@DOMAIN` into separate columns
* **`add_time_buckets`** — derives `time_day` and `time_hour` from the epoch-second `time` column

In [0]:
from pyspark.sql.functions import (
    col, when, lit, split, floor, current_timestamp
)

CATALOG = "workspace"
BRONZE = "bronze_lanl"
SILVER = "silver_lanl"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")
print(f"✓ Schema {CATALOG}.{SILVER} ready")

# --- helpers ---
def nullify_question_marks(df, cols):
    """Replace '?' sentinel values with proper NULLs."""
    for c in cols:
        df = df.withColumn(c, when(col(c) == "?", None).otherwise(col(c)))
    return df

def parse_user_domain(df, source_col, user_alias, domain_alias):
    """Split 'USER@DOMAIN' into separate columns."""
    parts = split(col(source_col), "@")
    return (df
        .withColumn(user_alias, parts.getItem(0))
        .withColumn(domain_alias, when(parts.getItem(1).isNotNull(), parts.getItem(1)))
    )

def add_time_buckets(df):
    """Add day and hour buckets from the epoch-second time column."""
    return (df
        .withColumn("time_day", floor(col("time") / 86400).cast("int"))
        .withColumn("time_hour", floor((col("time") % 86400) / 3600).cast("int"))
    )

print("✓ Helpers defined")

## 2. Silver Table Transforms

Each cell reads from `workspace.bronze_lanl`, applies cleaning and enrichment, and writes to `workspace.silver_lanl`. Key patterns:
* `?` → `NULL` across all string columns
* `user@domain` → parsed into separate user & domain columns
* Time bucketing for downstream aggregation
* Flows get a `service_type` classification from well-known port mappings

In [0]:
df_auth = spark.table(f"{CATALOG}.{BRONZE}.auth")

# Parse composite user@domain fields
df_auth = parse_user_domain(df_auth, "source_user", "src_user", "src_domain")
df_auth = parse_user_domain(df_auth, "destination_user", "dst_user", "dst_domain")

# Replace '?' with NULL
string_cols = ["source_user", "destination_user", "source_computer",
               "destination_computer", "auth_type", "logon_type",
               "auth_orientation", "success_failure"]
df_auth = nullify_question_marks(df_auth, string_cols)

# Derived columns
df_auth = (df_auth
    .withColumn("is_failed", when(col("success_failure") == "Fail", True).otherwise(False))
)
df_auth = add_time_buckets(df_auth)

# Select final column order
df_auth = df_auth.select(
    "time", "time_day", "time_hour",
    "source_user", "src_user", "src_domain",
    "destination_user", "dst_user", "dst_domain",
    "source_computer", "destination_computer",
    "auth_type", "logon_type", "auth_orientation",
    "success_failure", "is_failed"
)

df_auth.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.auth")
print(f"✓ auth -> {CATALOG}.{SILVER}.auth")

In [0]:
df_dns = spark.table(f"{CATALOG}.{BRONZE}.dns")

df_dns = nullify_question_marks(df_dns, ["source_computer", "resolved_computer"])
df_dns = add_time_buckets(df_dns)

df_dns = df_dns.select(
    "time", "time_day", "time_hour",
    "source_computer", "resolved_computer"
)

df_dns.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.dns")
print(f"✓ dns -> {CATALOG}.{SILVER}.dns")

In [0]:
from pyspark.sql.functions import create_map, coalesce

# Well-known port classification
PORT_MAP = {
    "80": "HTTP", "443": "HTTPS", "53": "DNS",
    "22": "SSH", "25": "SMTP", "110": "POP3",
    "143": "IMAP", "389": "LDAP", "445": "SMB",
    "3389": "RDP", "88": "Kerberos", "135": "RPC",
    "139": "NetBIOS", "636": "LDAPS", "993": "IMAPS",
    "995": "POP3S", "123": "NTP", "161": "SNMP",
    "5985": "WinRM", "5986": "WinRM-S"
}

port_map_expr = create_map([lit(x) for pair in PORT_MAP.items() for x in pair])

df_flows = spark.table(f"{CATALOG}.{BRONZE}.flows")

df_flows = nullify_question_marks(df_flows, [
    "source_computer", "source_port",
    "destination_computer", "destination_port"
])
df_flows = add_time_buckets(df_flows)

# Classify destination port into a human-readable service type
df_flows = df_flows.withColumn(
    "service_type",
    coalesce(port_map_expr[col("destination_port")], lit("Other"))
)

df_flows = df_flows.select(
    "time", "time_day", "time_hour",
    "duration", "source_computer", "source_port",
    "destination_computer", "destination_port",
    "service_type", "protocol", "packet_count", "byte_count"
)

df_flows.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.flows")
print(f"✓ flows -> {CATALOG}.{SILVER}.flows")

In [0]:
df_proc = spark.table(f"{CATALOG}.{BRONZE}.proc")

df_proc = parse_user_domain(df_proc, "user", "proc_user", "proc_domain")
df_proc = nullify_question_marks(df_proc, ["user", "computer", "process_name", "start_end"])
df_proc = add_time_buckets(df_proc)

df_proc = df_proc.withColumn(
    "is_start", when(col("start_end") == "Start", True).otherwise(False)
)

df_proc = df_proc.select(
    "time", "time_day", "time_hour",
    "user", "proc_user", "proc_domain",
    "computer", "process_name",
    "start_end", "is_start"
)

df_proc.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.proc")
print(f"✓ proc -> {CATALOG}.{SILVER}.proc")

In [0]:
df_redteam = spark.table(f"{CATALOG}.{BRONZE}.redteam")

df_redteam = parse_user_domain(df_redteam, "user", "rt_user", "rt_domain")
df_redteam = add_time_buckets(df_redteam)

df_redteam = df_redteam.select(
    "time", "time_day", "time_hour",
    "user", "rt_user", "rt_domain",
    "source_computer", "destination_computer"
)

df_redteam.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.redteam")
print(f"✓ redteam -> {CATALOG}.{SILVER}.redteam")

## 3. Labeled Dataset Construction

Joins `silver_lanl.auth` with `silver_lanl.redteam` to produce `auth_labeled` — the primary table for ML, with an `is_compromised` boolean flag. Note the slight row fan-out due to many-to-one redteam key groups (see note below).

### 📝 auth_labeled row-count note

**Observed:** `auth_labeled` (1,051,430,496) has **37 more rows** than `auth` (1,051,430,459).  
**Cause:** The left join between `auth` and `redteam` on `(time, source_user, source_computer, destination_computer)` is **many-redteam-to-one-auth** in 12 key groups. Multiple redteam entries share identical join keys (same second, user, and computer pair), so each matching auth row fans out. The largest group is `U8946@DOM1` at time 1233528 on `C17693→C19038` with **9 duplicate redteam entries**.

**Interpretation:** The red team logged repeated compromise events at the same second — likely repeated lateral movement attempts or tool re-executions during the engagement.

**Rationale for keeping as-is:** Each redteam record represents a discrete compromise event logged by the red team. Deduplicating would discard valid attack labels. The 37 extra rows (∼0.000004% of the dataset) preserve full red team fidelity, which is preferable for downstream ML where recall on the minority class matters.

In [0]:
# Read silver tables
df_auth = spark.table(f"{CATALOG}.{SILVER}.auth")
df_rt   = (spark.table(f"{CATALOG}.{SILVER}.redteam")
    .select(
        col("time").alias("rt_time"),
        col("user").alias("rt_user_key"),
        col("source_computer").alias("rt_src"),
        col("destination_computer").alias("rt_dst")
    )
)

# Left-join auth with redteam on matching keys
join_cond = (
    (df_auth.time == df_rt.rt_time) &
    (df_auth.source_user == df_rt.rt_user_key) &
    (df_auth.source_computer == df_rt.rt_src) &
    (df_auth.destination_computer == df_rt.rt_dst)
)

df_labeled = (df_auth
    .join(df_rt, join_cond, "left")
    .withColumn("is_compromised", col("rt_time").isNotNull())
    .drop("rt_time", "rt_user_key", "rt_src", "rt_dst")
)

df_labeled.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.auth_labeled")
print(f"✓ auth_labeled -> {CATALOG}.{SILVER}.auth_labeled")

## 4. Validation

Row counts for all silver tables, including `auth_labeled`.

In [0]:
tables = ["auth", "dns", "flows", "proc", "redteam", "auth_labeled"]
counts = []
for t in tables:
    n = spark.table(f"{CATALOG}.{SILVER}.{t}").count()
    counts.append((t, n))

df_counts = spark.createDataFrame(counts, ["table", "row_count"])
display(df_counts)